# Reproducibility notes

Run this notebook from top to bottom. All paths are relative to the project folder, random behaviour is controlled by `RANDOM_SEED`, and generated files are written to `outputs/`. See `README.md` for the required folder structure and complete run instructions.


# COMP5310 Assignment 1
## 1. Comparative Analysis


### Setup and airline dataset

The following cell imports all dependencies once, fixes the project-wide random seed, creates the output folder, and loads data through one documented relative-path convention.


In [1]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.preprocessing import MinMaxScaler

RANDOM_SEED = 5310
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.figsize": (9, 5), "axes.titleweight": "bold"})

def load_csv(filename):
    # Load a required CSV and raise an actionable error if it is absent.
    path = DATA_DIR / filename
    if not path.exists():
        raise FileNotFoundError(
            f"Required file not found: {path}. "
            "See README.md for the expected project structure."
        )
    return pd.read_csv(path)

print("pandas version:", pd.__version__)
print("Random seed:", RANDOM_SEED)

airline_delay= pd.read_csv('airline_delay.csv')


pandas version: 3.0.3
Random seed: 5310


FileNotFoundError: [Errno 2] No such file or directory: 'airline_delay.csv'

In [ ]:
display(airline_delay.head())

In [ ]:
display(airline_delay.describe(include='all'))

In [ ]:
dtype_df = pd.DataFrame(
    airline_delay.dtypes,
    columns=["Data Type"]
)

display(dtype_df)

In [ ]:
missing_df = pd.DataFrame({
    "Missing Count": airline_delay.isnull().sum(),
    "Missing %": (
        airline_delay.isnull().sum()
        / len(airline_delay)
        * 100
    ).round(2)
})

display(missing_df)

In [ ]:
duplicate_count = airline_delay.duplicated().sum()

print(f"Duplicate Rows: {duplicate_count}")

In [ ]:
summary = pd.DataFrame({
    "Rows": [airline_delay.shape[0]],
    "Columns": [airline_delay.shape[1]],
    "Missing Values": [airline_delay.isnull().sum().sum()],
    "Duplicate Rows": [airline_delay.duplicated().sum()],
    "Numeric Variables": [
        len(airline_delay.select_dtypes(include='number').columns)
    ],
    "Categorical Variables": [
        len(airline_delay.select_dtypes(
            include=['object','string','category']
        ).columns)
    ]
})

display(summary)

In [ ]:


# Count missing values
missing = airline_delay.isnull().sum()

# Keep only columns with missing values
missing = missing[missing > 0].sort_values(ascending=False)

plt.figure(figsize=(10, 5))
missing.plot(kind='bar')
plt.title('Missing Values by Column')
plt.xlabel('Columns')
plt.ylabel('Number of Missing Values')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Railway dataset

The railway data is loaded from the same `data/` directory used for every raw dataset.


In [ ]:
train_occupancy= pd.read_csv('train_occupancy.csv')


In [ ]:
display(train_occupancy.head())

In [ ]:
display(train_occupancy.describe(include='all'))

In [ ]:
dtype_df = pd.DataFrame(
    train_occupancy.dtypes,
    columns=["Data Type"]
)
display(dtype_df)

In [ ]:
missing_df = pd.DataFrame({
    "Missing Count": train_occupancy.isnull().sum(),
    "Missing %": (
        train_occupancy.isnull().sum()
        / len(train_occupancy)
        * 100
    ).round(2)
})
display(missing_df)

In [ ]:
duplicate_count = train_occupancy.duplicated().sum()

print(f"Duplicate Rows: {duplicate_count}")

In [ ]:
summary = pd.DataFrame({
    "Rows": [train_occupancy.shape[0]],
    "Columns": [train_occupancy.shape[1]],
    "Missing Values": [train_occupancy.isnull().sum().sum()],
    "Duplicate Rows": [train_occupancy.duplicated().sum()],
    "Numeric Variables": [
        len(train_occupancy.select_dtypes(include='number').columns)
    ],
    "Categorical Variables": [
        len(train_occupancy.select_dtypes(
            include=['object','string','category']
        ).columns)
    ]
})

display(summary)

In [ ]:

# Count missing values
missing = train_occupancy.isnull().sum()

# Keep only columns with missing values
missing = missing[missing > 0].sort_values(ascending=False)

plt.figure(figsize=(10, 5))
missing.plot(kind='bar')
plt.title('Missing Values by Column')
plt.xlabel('Columns')
plt.ylabel('Number of Missing Values')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Hotel bookings dataset

The selected hotel dataset is loaded once here and reused by Parts 2–4.


In [ ]:
hotel_bookings= pd.read_csv('hotel_bookings.csv')

In [ ]:
display(hotel_bookings.head())

In [ ]:
display(hotel_bookings.describe(include='all'))

In [ ]:
dtype_df = pd.DataFrame(
    hotel_bookings.dtypes,
    columns=["Data Type"]
)

display(dtype_df)

In [ ]:
missing_df = pd.DataFrame({
    "Missing Count": hotel_bookings.isnull().sum(),
    "Missing %": (
        hotel_bookings.isnull().sum()
        / len(hotel_bookings)
        * 100
    ).round(2)
})

display(missing_df)

In [ ]:
duplicate_count = hotel_bookings.duplicated().sum()

print(f"Duplicate Rows: {duplicate_count}")

In [ ]:
summary = pd.DataFrame({
    "Rows": [hotel_bookings.shape[0]],
    "Columns": [hotel_bookings.shape[1]],
    "Missing Values": [hotel_bookings.isnull().sum().sum()],
    "Duplicate Rows": [hotel_bookings.duplicated().sum()],
    "Numeric Variables": [
        len(hotel_bookings.select_dtypes(include='number').columns)
    ],
    "Categorical Variables": [
        len(hotel_bookings.select_dtypes(
            include=['object','string','category']
        ).columns)
    ]
})

display(summary)

## Dataset Comparison and Selection
Comparison of the 3 datasets 
`Airline_delay` , `Train_occupancy` & `Hotel_bookings`
to check whether that data is useful to solve the given problem statement depending on the quality of data, missing values, duplicate rows , usable rows and uniqueness ratio.


In [ ]:
datasets = {
    "Airline Delay": airline_delay,
    "Train Occupancy": train_occupancy,
    "Hotel Booking": hotel_bookings
}

comparison = []

for name, df in datasets.items():

    rows = len(df)
    cols = len(df.columns)

    total_missing = df.isnull().sum().sum()
    missing_pct = round(total_missing / (rows * cols) * 100, 2)

    duplicates = df.duplicated().sum()
    duplicate_pct = round(duplicates / rows * 100, 2)

    usable_rows = len(df.drop_duplicates().dropna())

    comparison.append({
        "Dataset": name,
        "Rows": rows,
        "Columns": cols,
        "Missing %": missing_pct,
        "Duplicate %": duplicate_pct,
        "Numeric Vars": len(df.select_dtypes(include=np.number).columns),
        "Categorical Vars": len(df.select_dtypes(include=['object','string','category']).columns),
        "Unique Ratio": round(df.nunique().mean() / rows * 100, 2)
    })

comparison_df = pd.DataFrame(comparison)


comparison_df["Quality Score"] = (
    100
    - comparison_df["Missing %"]
    - comparison_df["Duplicate %"]
)
comparison_df["Modelling Score"] = (
    comparison_df["Rows"] / comparison_df["Rows"].max() * 30
    + comparison_df["Numeric Vars"] / comparison_df["Numeric Vars"].max() * 30
    + comparison_df["Categorical Vars"] / comparison_df["Categorical Vars"].max() * 20
    
)
comparison_df["Feature Richness"] = (
    comparison_df["Numeric Vars"] +
    comparison_df["Categorical Vars"]
)
comparison_df[["Dataset","Quality Score"]]

display(comparison_df)

Trying Different Types of plots to give a suitable graphical summary which will make our selection more simpler and comprehesive

Below trying to plot a graph based on the missing and duplicate values.

Comparison scorecard that plots the rows , usable % , quality score , numeric values , categorical values to analyse the strength of the data and how functional it is for further analysis.

In [ ]:

metrics = comparison_df[
    [
        "Rows",
        "Quality Score",
        "Numeric Vars",
        "Categorical Vars"
    ]
].copy()

scaled = pd.DataFrame(
    MinMaxScaler().fit_transform(metrics),
    columns=metrics.columns,
    index=comparison_df["Dataset"]
)

scaled *= 100
ax = scaled.plot(
    kind="bar",
    figsize=(12,6)
)

plt.title("Dataset Comparison Scorecard")
plt.ylabel("Normalised Score (0-100)")
plt.xticks(rotation=0)
plt.legend(bbox_to_anchor=(1.05,1))
plt.tight_layout()
plt.show()


Overall Comparison of the content that includes the rows, feature richness, usable, quality scores


In [ ]:
comparison_df["Suitability Score"] = (
    0.35 * (comparison_df["Rows"] / comparison_df["Rows"].max() * 100)
    + 0.30 * (comparison_df["Feature Richness"] / comparison_df["Feature Richness"].max() * 100)
    + 0.15 * comparison_df["Quality Score"]
)

comparison_df = comparison_df.sort_values(
    "Suitability Score",
    ascending=False
)

plt.figure(figsize=(8,5))
bars = plt.bar(
    comparison_df["Dataset"],
    comparison_df["Suitability Score"]
)

plt.title("Overall Dataset Suitability")
plt.ylabel("Suitability Score")

for bar in bars:
    plt.text(
        bar.get_x()+bar.get_width()/2,
        bar.get_height()+1,
        f"{bar.get_height():.1f}",
        ha='center'
    )

plt.show()

## Conclusion

I have analysed all the tables based on the missing values and the quality of the data and even though the table `Train_occupancy` has less missing values and requires less cleaning the quality of data is not that great from a statistical point of view. The train dataset is also very small with very less comprehensive data for further analsyis. In conclusion the `Hotel_bookings` data has better numerical and categorical data which improves the quality of the data even though the missing values and duplication percentage is really high. The data also missing values in the columns which might not deeply affect our analysis and the corelation to our goal is very insignificant.